In [ ]:
!pip install sahi ultralytics -q
print("Installed")

In [ ]:
from pathlib import Path

MODEL_PATH   = "/kaggle/input/models/natair/chicken-yolo8m-model/pytorch/default/1/best.pt"
FRAMES_DIR   = "/kaggle/input/datasets/natair/chicken-frames/chicken-frames"
OUTPUT_DIR   = "/kaggle/working"
IMG_SIZE     = 1280
CONF_THRESH  = 0.45
LABEL_NAME   = "chicken"

# SAHI slicing parameters
SLICE_SIZE     = 640      # size of a single tile (in pixels)
OVERLAP_RATIO  = 0.2      # 20% overlap between tiles

In [ ]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
import cv2
from tqdm import tqdm

assert Path(MODEL_PATH).exists(), f"Model not found: {MODEL_PATH}"

detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=MODEL_PATH,
    confidence_threshold=CONF_THRESH,
    device="cuda",
)

print("Model uploaded in SAHI")

In [ ]:
# SAHI cuts each frame into tiles, then performs detection on each one, then merges the results back together
EXTS = {'.jpg', '.jpeg', '.png'}
frame_files = sorted([
    p for p in Path(FRAMES_DIR).rglob("*") if p.suffix.lower() in EXTS
])
print(f"Found frames: {len(frame_files)}")

all_results = {}   # {frame_name: [[x1,y1,x2,y2,conf], ...]}

for fp in tqdm(frame_files, desc="SAHI detection"):
    result = get_sliced_prediction(
        str(fp),
        detection_model,
        slice_height=SLICE_SIZE,
        slice_width=SLICE_SIZE,
        overlap_height_ratio=OVERLAP_RATIO,
        overlap_width_ratio=OVERLAP_RATIO,
        verbose=0,
    )
    boxes = []
    for obj in result.object_prediction_list:
        bbox = obj.bbox
        boxes.append([bbox.minx, bbox.miny, bbox.maxx, bbox.maxy, obj.score.value])
    all_results[fp.stem] = boxes

total_det = sum(len(v) for v in all_results.values())
avg_det   = total_det / len(all_results)
print(f"\n Ready. Overall detections: {total_det}")
print(f" Average for a frame: {avg_det:.1f}")

In [ ]:
# Writing .txt files in YOLO format alongside the images
labels_dir = Path(OUTPUT_DIR) / "auto_labels"
labels_dir.mkdir(exist_ok=True)

for fp in frame_files:
    img = cv2.imread(str(fp))
    H, W = img.shape[:2]
    boxes = all_results[fp.stem]

    lines = []
    for x1, y1, x2, y2, conf in boxes:
        cx = (x1 + x2) / 2 / W
        cy = (y1 + y2) / 2 / H
        w  = (x2 - x1) / W
        h  = (y2 - y1) / H
        lines.append(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

    out_txt = labels_dir / f"{fp.stem}.txt"
    out_txt.write_text("\n".join(lines))

print(f" Labels saved: {labels_dir}")
print(f" Files: {len(list(labels_dir.glob('*.txt')))}")

# obj.names for CVAT YOLO import
(Path(OUTPUT_DIR) / "obj.names").write_text(LABEL_NAME + "\n")

In [ ]:
# Importing as xml for CVAT
import xml.etree.ElementTree as ET
from xml.dom import minidom

root = ET.Element("annotations")
ET.SubElement(root, "version").text = "1.1"
meta = ET.SubElement(root, "meta")
task = ET.SubElement(meta, "task")
ET.SubElement(task, "size").text = str(len(frame_files))
lbls = ET.SubElement(task, "labels")
lbl  = ET.SubElement(lbls, "label")
ET.SubElement(lbl, "name").text = LABEL_NAME

for idx, fp in enumerate(frame_files):
    img_el = ET.SubElement(root, "image", {
        "id": str(idx), "name": fp.name,
    })
    for x1, y1, x2, y2, conf in all_results[fp.stem]:
        ET.SubElement(img_el, "box", {
            "label": LABEL_NAME,
            "xtl": f"{x1:.2f}", "ytl": f"{y1:.2f}",
            "xbr": f"{x2:.2f}", "ybr": f"{y2:.2f}",
            "occluded": "0", "z_order": "0",
        })

xml_str = minidom.parseString(ET.tostring(root)).toprettyxml(indent="  ")
out_xml = Path(OUTPUT_DIR) / "auto_annotations_cvat.xml"
out_xml.write_text(xml_str, encoding="utf-8")
print(f"CVAT XML saved: {out_xml}")

In [ ]:
import matplotlib.pyplot as plt

sample_indices = [0, len(frame_files)//4, len(frame_files)//2,
                   3*len(frame_files)//4, len(frame_files)-1]

fig, axes = plt.subplots(1, 5, figsize=(25, 6))
for ax, idx in zip(axes, sample_indices):
    fp = frame_files[idx]
    img = cv2.cvtColor(cv2.imread(str(fp)), cv2.COLOR_BGR2RGB)
    n_boxes = 0
    for x1, y1, x2, y2, conf in all_results[fp.stem]:
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (0,255,0), 2)
        n_boxes += 1
    ax.imshow(img)
    ax.set_title(f"{fp.name}\n{n_boxes} detections", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/sahi_quality_check.jpg", dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved: {OUTPUT_DIR}/sahi_quality_check.jpg")

In [ ]:
import shutil

archive_base = f"{OUTPUT_DIR}/chicken_autolabel_results"

bundle_dir = Path(OUTPUT_DIR) / "_bundle"
bundle_dir.mkdir(exist_ok=True)

shutil.copytree(labels_dir, bundle_dir / "auto_labels", dirs_exist_ok=True)
shutil.copy(Path(OUTPUT_DIR) / "obj.names", bundle_dir / "obj.names")
shutil.copy(out_xml, bundle_dir / "auto_annotations_cvat.xml")
shutil.copy(f"{OUTPUT_DIR}/sahi_quality_check.jpg", bundle_dir / "sahi_quality_check.jpg")

shutil.make_archive(archive_base, "zip", bundle_dir)

shutil.rmtree(bundle_dir)

archive_path = f"{archive_base}.zip"
archive_size_mb = Path(archive_path).stat().st_size / 1e6

print(f" Created archive: {archive_path}")
print(f" Size: {archive_size_mb:.1f} MB")